# Домашняя работа 1 — Бенчмарк форматов хранения данных

**Датасет:** Amazon Reviews 2023 (Books) — отзывы покупателей с Amazon за 1996–2023 годы.  
Покупатели оставляли отзывы на товары: текст, оценка от 1 до 5, дата и т.д.  
Полный датасет содержит 571 млн отзывов (~18.7 ГБ для категории Books). Для работы берём 3 млн строк.

**Источник:** https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

### Описание признаков (features)

| Признак | Тип | Описание |
|---|---|---|
| `rating` | float | Оценка товара от 1.0 до 5.0 |
| `title` | str | Краткий заголовок отзыва |
| `text` | str | Полный текст отзыва |
| `asin` | str | Уникальный ID товара на Amazon |
| `parent_asin` | str | ID родительского товара (для вариаций) |
| `user_id` | str | Анонимизированный ID автора отзыва |
| `timestamp` | datetime | Дата и время публикации отзыва |
| `helpful_vote` | int | Количество голосов "полезный отзыв" |
| `verified_purchase` | bool | Подтверждённая покупка (True/False) |

**Целевой признак:** `rating` — оценка товара, которую ставит покупатель.

In [1]:
!pip install pyarrow fastparquet duckdb polars sqlalchemy orjson ujson numba -q

In [2]:
import os
import csv
import json
import time
import sqlite3
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

# data processing
import numpy as np
import pandas as pd
import requests

# columnar formats
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# analytics engines
import duckdb
import polars as pl
from sqlalchemy import create_engine, text

# JIT compilation
from numba import njit

# JSON parsers
import orjson
import ujson

In [3]:
DATASET_URL = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Books.jsonl"
TARGET_ROWS = 3_000_000
CHUNK_SIZE = 500_000

## 1. Загрузка и подготовка данных

In [4]:
r = requests.head(DATASET_URL, allow_redirects=True)
size_gb = int(r.headers.get("Content-Length", 0)) / (1024**3)
print(f"Full file size: {size_gb:.1f} GB")

Full file size: 18.7 GB


In [5]:
# stream-download rows (avoids loading the entire file into memory)
rows = []

with requests.get(DATASET_URL, stream=True) as r:
    for i, line in enumerate(r.iter_lines()):
        if i >= TARGET_ROWS:
            break
        rows.append(json.loads(line))
        if (i + 1) % 500_000 == 0:
            print(f"Loaded {i + 1:,} rows...")

print(f"Done! Total rows: {len(rows):,}")

# build DataFrame and clean up
df = pd.DataFrame(rows)
del rows

df = df.drop(columns=["images"])
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
df["rating"] = df["rating"].astype("float32")

print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")
print(f"Columns: {list(df.columns)}")
df.head(5)

Loaded 500,000 rows...
Loaded 1,000,000 rows...
Loaded 1,500,000 rows...
Loaded 2,000,000 rows...
Loaded 2,500,000 rows...
Loaded 3,000,000 rows...
Done! Total rows: 3,000,000
Memory usage: 2.37 GB
Columns: ['rating', 'title', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1.0,Not a watercolor book! Seems like copies imo.,It is definitely not a watercolor book. The p...,B09BGPFTDB,B09BGPFTDB,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2022-01-17 06:06:38.485,0,True
1,5.0,Updated: after 1st arrived damaged this one is...,Updated: after first book arrived very damaged...,0593235657,0593235657,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2021-12-27 18:26:44.904,1,True
2,5.0,Excellent! I love it!,I bought it for the bag on the front so it pai...,1782490671,1782490671,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2021-12-24 22:04:55.102,0,True
3,5.0,Updated after 1st arrived damaged. Excellent,Updated: after 1st arrived damaged the replace...,0593138228,0593138228,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2021-12-24 16:55:06.602,0,False
4,5.0,Beautiful patterns!,I love this book! The patterns are lovely. I ...,0823098079,0823098079,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2021-11-19 08:57:33.230,0,True


## 2. Бенчмарк записи в разные форматы

Сохраняем один и тот же DataFrame в 7 форматов и замеряем wall-time (реальное время) и cpu-time.

In [6]:
results_write = []


def bench_write(name, func):
    t0_wall = time.time()
    t0_cpu = time.process_time()
    func()
    wall = time.time() - t0_wall
    cpu = time.process_time() - t0_cpu
    size = 0
    if os.path.exists(f"data.{name}"):
        size = os.path.getsize(f"data.{name}") / (1024**3)
    results_write.append({
        "format": name,
        "wall_time": round(wall, 2),
        "cpu_time": round(cpu, 2),
        "size_gb": round(size, 2),
    })
    print(f"{name:10s} | wall: {wall:7.2f}s | cpu: {cpu:7.2f}s | size: {size:.2f} GB")


def write_json_chunked():
    """Write JSON in chunks to avoid MemoryError on large datasets."""
    with open("data.json", "w", encoding="utf-8") as f:
        f.write("[\n")
        for start in range(0, len(df), CHUNK_SIZE):
            part = df.iloc[start:start + CHUNK_SIZE].to_json(
                orient="records", force_ascii=False
            )
            if start > 0:
                f.write(",\n")
            f.write(part[1:-1])
        f.write("\n]")


def write_jsonl_chunked():
    """Write JSONL in chunks to avoid MemoryError on large datasets."""
    with open("data.jsonl", "w", encoding="utf-8") as f:
        for start in range(0, len(df), CHUNK_SIZE):
            part = df.iloc[start:start + CHUNK_SIZE].to_json(
                orient="records", lines=True, force_ascii=False
            )
            f.write(part)


def write_sqlite():
    conn = sqlite3.connect("data.sqlite")
    df.to_sql("reviews", conn, if_exists="replace", index=False)
    conn.close()


print(f"{'Format':10s} | {'Wall time':>10s} | {'CPU time':>10s} | {'Size':>10s}")
print("-" * 55)

bench_write("csv", lambda: df.to_csv("data.csv", index=False))
bench_write("tsv", lambda: df.to_csv("data.tsv", sep="\t", index=False))
bench_write("json", write_json_chunked)
bench_write("jsonl", write_jsonl_chunked)
bench_write("parquet", lambda: df.to_parquet("data.parquet", index=False))
bench_write("orc", lambda: df.to_orc("data.orc", index=False))
bench_write("sqlite", write_sqlite)

Format     |  Wall time |   CPU time |       Size
-------------------------------------------------------
csv        | wall:   21.12s | cpu:   21.11s | size: 1.54 GB
tsv        | wall:   20.98s | cpu:   20.94s | size: 1.53 GB
json       | wall:   14.56s | cpu:   13.53s | size: 1.82 GB
jsonl      | wall:   25.41s | cpu:   24.34s | size: 1.83 GB
parquet    | wall:    7.52s | cpu:    7.50s | size: 0.87 GB
orc        | wall:    4.19s | cpu:    4.06s | size: 1.45 GB
sqlite     | wall:   38.13s | cpu:   18.73s | size: 1.79 GB


**Вывод по записи:** Parquet и ORC — лидеры: пишутся за 4-8 секунд и занимают меньше всего места (Parquet — 0.87 GB, почти в 2 раза компактнее JSON). CSV/TSV и JSON — середина (~14-21s). SQLite — самый медленный на запись (38.13s), потому что INSERT в базу данных идёт построчно с транзакциями. JSONL оказался медленнее обычного JSON из-за чанковой записи.

## 3. Бенчмарк чтения из разных форматов

In [7]:
results_read = []


def bench_read(name, func):
    t0_wall = time.time()
    t0_cpu = time.process_time()
    data = func()
    wall = time.time() - t0_wall
    cpu = time.process_time() - t0_cpu
    results_read.append({
        "format": name,
        "wall_time": round(wall, 2),
        "cpu_time": round(cpu, 2),
    })
    print(f"{name:10s} | wall: {wall:7.2f}s | cpu: {cpu:7.2f}s | rows: {len(data):,}")
    del data


def read_jsonl_chunked():
    chunks = pd.read_json("data.jsonl", lines=True, chunksize=CHUNK_SIZE)
    return pd.concat(chunks, ignore_index=True)


print(f"{'Format':10s} | {'Wall time':>10s} | {'CPU time':>10s} | {'Rows':>10s}")
print("-" * 55)

bench_read("csv", lambda: pd.read_csv("data.csv"))
bench_read("tsv", lambda: pd.read_csv("data.tsv", sep="\t"))
bench_read("json", lambda: pd.read_json("data.json"))
bench_read("jsonl", read_jsonl_chunked)
bench_read("parquet", lambda: pd.read_parquet("data.parquet"))
bench_read("orc", lambda: pd.read_orc("data.orc"))
bench_read("sqlite", lambda: pd.read_sql(
    "SELECT * FROM reviews", sqlite3.connect("data.sqlite")
))

Format     |  Wall time |   CPU time |       Rows
-------------------------------------------------------
csv        | wall:   20.04s | cpu:   20.05s | rows: 3,000,000
tsv        | wall:   20.57s | cpu:   20.58s | rows: 3,000,000
json       | wall:   22.97s | cpu:   23.20s | rows: 3,000,000
jsonl      | wall:   31.16s | cpu:   31.05s | rows: 3,000,000
parquet    | wall:    6.39s | cpu:    7.06s | rows: 3,000,000
orc        | wall:    7.22s | cpu:    5.86s | rows: 3,000,000
sqlite     | wall:   11.90s | cpu:   10.56s | rows: 3,000,000


**Вывод по чтению:** Parquet (6.4s) и ORC (7.2s) читаются в 3-4 раза быстрее текстовых форматов. SQLite (11.9s) тоже быстрый благодаря внутренней оптимизации движка. JSONL — самый медленный (31.2s), так как pandas парсит каждую строку отдельно. CSV/TSV — примерно одинаково (~20s), JSON чуть медленнее (23s)

## 4. SQL-подобные запросы через разные библиотеки

На каждом инструменте выполняем 3 типа запросов:
1. **Фильтрация** — отзывы с оценкой 5 за 2022 год
2. **Группировка** — среднее и количество отзывов по годам
3. **Оконная функция** — топ-3 самых полезных отзыва за каждый год

### 4.1 DuckDB

In [8]:
con = duckdb.connect()
con.execute("CREATE TABLE reviews AS SELECT * FROM read_parquet('data.parquet')")

# filter by predicate: 5-star reviews from 2022
q1 = con.execute("""
    SELECT title, text, timestamp
    FROM reviews
    WHERE rating = 5 AND YEAR(timestamp) = 2022
    LIMIT 5
""").fetchdf()
print("=== Filter: rating=5, year=2022 ===")
print(q1, "\n")

# group by year: count and average rating
q2 = con.execute("""
    SELECT YEAR(timestamp) AS year,
           COUNT(*) AS cnt,
           ROUND(AVG(rating), 2) AS avg_rating
    FROM reviews
    GROUP BY year
    ORDER BY year
""").fetchdf()
print("=== Group by: stats per year ===")
print(q2, "\n")

# window function: top-3 most helpful reviews per year
q3 = con.execute("""
    SELECT title, helpful_vote, YEAR(timestamp) AS year,
           RANK() OVER (
               PARTITION BY YEAR(timestamp)
               ORDER BY helpful_vote DESC
           ) AS rank
    FROM reviews
    QUALIFY rank <= 3
    ORDER BY year, rank
""").fetchdf()
print("=== Window: top-3 helpful reviews per year ===")
print(q3)

con.close()

=== Filter: rating=5, year=2022 ===
                                               title  \
0  A sweet story with a positive message for any ...   
1                                           So cute!   
2                        Good to the last gory drop!   
3                       What rock have I been under?   
4          A short read but jam packed with goodness   

                                                text               timestamp  
0  This is such a sweet and thoughtful story. No ... 2022-02-10 04:01:49.024  
1  Simple message and little writing but so cute ... 2022-03-18 04:25:53.809  
2  The imagery in this book is so good that I act... 2022-06-19 00:57:19.827  
3  I've seen all the Hell Raiser movies but some ... 2022-06-13 04:59:16.252  
4  This is an amazing book that makes me look at ... 2022-06-11 05:09:43.375   

=== Group by: stats per year ===
    year     cnt  avg_rating
0   1996      12        4.92
1   1997     169        4.35
2   1998     866        4.23
3 

### 4.2 Polars

df_pl = pl.read_parquet("data.parquet")

# 1. filter by predicate
q1 = df_pl.filter(
    (pl.col("rating") == 5) & (pl.col("timestamp").dt.year() == 2022)
).select("title", "text", "timestamp").head(5)
print("=== Filter: rating=5, year=2022 ===")
print(q1, "\n")

# 2. group by year
q2 = df_pl.group_by(
    pl.col("timestamp").dt.year().alias("year")
).agg(
    pl.len().alias("cnt"),
    pl.col("rating").mean().round(2).alias("avg_rating"),
).sort("year")
print("=== Group by: stats per year ===")
print(q2, "\n")

# 3. window function: rank by helpful_vote within each year
q3 = df_pl.with_columns(
    pl.col("helpful_vote")
    .rank(method="min", descending=True)
    .over(pl.col("timestamp").dt.year())
    .alias("rank")
).filter(
    pl.col("rank") <= 3
).select(
    "title", "helpful_vote",
    pl.col("timestamp").dt.year().alias("year"),
    "rank",
).sort("year", "rank")
print("=== Window: top-3 helpful reviews per year ===")
print(q3)

### 4.3 SQLAlchemy (SQLite)

In [10]:
engine = create_engine("sqlite:///data.sqlite")

# 1. filter by predicate
with engine.connect() as conn:
    q1 = pd.read_sql(text("""
        SELECT title, text, timestamp
        FROM reviews
        WHERE rating = 5 AND timestamp LIKE '2022%'
        LIMIT 5
    """), conn)
print("=== Filter: rating=5, year=2022 ===")
print(q1, "\n")

# 2. group by year
with engine.connect() as conn:
    q2 = pd.read_sql(text("""
        SELECT SUBSTR(timestamp, 1, 4) AS year,
               COUNT(*) AS cnt,
               ROUND(AVG(rating), 2) AS avg_rating
        FROM reviews
        GROUP BY year
        ORDER BY year
    """), conn)
print("=== Group by: stats per year ===")
print(q2, "\n")

# 3. window function: top-3 most helpful reviews per year
with engine.connect() as conn:
    q3 = pd.read_sql(text("""
        SELECT title, helpful_vote, year, rank
        FROM (
            SELECT title, helpful_vote,
                   SUBSTR(timestamp, 1, 4) AS year,
                   RANK() OVER (
                       PARTITION BY SUBSTR(timestamp, 1, 4)
                       ORDER BY helpful_vote DESC
                   ) AS rank
            FROM reviews
        )
        WHERE rank <= 3
        ORDER BY year, rank
    """), conn)
print("=== Window: top-3 helpful reviews per year ===")
print(q3)

=== Filter: rating=5, year=2022 ===
                                               title  \
0  A sweet story with a positive message for any ...   
1                                           So cute!   
2                        Good to the last gory drop!   
3                       What rock have I been under?   
4          A short read but jam packed with goodness   

                                                text  \
0  This is such a sweet and thoughtful story. No ...   
1  Simple message and little writing but so cute ...   
2  The imagery in this book is so good that I act...   
3  I've seen all the Hell Raiser movies but some ...   
4  This is an amazing book that makes me look at ...   

                    timestamp  
0  2022-02-10 04:01:49.024000  
1  2022-03-18 04:25:53.809000  
2  2022-06-19 00:57:19.827000  
3  2022-06-13 04:59:16.252000  
4  2022-06-11 05:09:43.375000   

=== Group by: stats per year ===
    year     cnt  avg_rating
0   1996      12        4.92
1   1

### 4.4 PyArrow

In [11]:
table = pq.read_table("data.parquet")

# 1. filter by predicate using pyarrow compute
mask = pc.and_(
    pc.equal(table.column("rating"), 5),
    pc.equal(pc.year(table.column("timestamp")), 2022),
)
filtered = table.filter(mask).slice(0, 5).select(["title", "text", "timestamp"])
print("=== Filter: rating=5, year=2022 ===")
print(filtered.to_pandas(), "\n")

# 2. group by year (convert to pandas for aggregation)
df_arrow = table.to_pandas()
df_arrow["year"] = df_arrow["timestamp"].dt.year
grouped = df_arrow.groupby("year").agg(
    cnt=("rating", "count"),
    avg_rating=("rating", "mean"),
).round(2).reset_index()
print("=== Group by: stats per year ===")
print(grouped, "\n")

# 3. window function: rank within each year
df_arrow["rank"] = df_arrow.groupby("year")["helpful_vote"].rank(
    method="min", ascending=False
)
top3 = (
    df_arrow[df_arrow["rank"] <= 3][["title", "helpful_vote", "year", "rank"]]
    .sort_values(["year", "rank"])
)
print("=== Window: top-3 helpful reviews per year ===")
print(top3)

del df_arrow

=== Filter: rating=5, year=2022 ===
                                               title  \
0  A sweet story with a positive message for any ...   
1                                           So cute!   
2                        Good to the last gory drop!   
3                       What rock have I been under?   
4          A short read but jam packed with goodness   

                                                text               timestamp  
0  This is such a sweet and thoughtful story. No ... 2022-02-10 04:01:49.024  
1  Simple message and little writing but so cute ... 2022-03-18 04:25:53.809  
2  The imagery in this book is so good that I act... 2022-06-19 00:57:19.827  
3  I've seen all the Hell Raiser movies but some ... 2022-06-13 04:59:16.252  
4  This is an amazing book that makes me look at ... 2022-06-11 05:09:43.375   

=== Group by: stats per year ===
    year     cnt  avg_rating
0   1996      12        4.92
1   1997     169        4.35
2   1998     866        4.23
3 

### 4.5 Raw Python (без библиотек)

In [12]:
# 1. filter by predicate
print("=== Filter: rating=5, year=2022 ===")
count = 0
with open("data.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row["rating"] == "5.0" and row["timestamp"].startswith("2022"):
            print(f"  {row['title'][:60]}  |  {row['timestamp']}")
            count += 1
            if count >= 5:
                break

# 2. group by year
stats = defaultdict(lambda: {"cnt": 0, "total": 0.0})

with open("data.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        year = row["timestamp"][:4]
        stats[year]["cnt"] += 1
        stats[year]["total"] += float(row["rating"])

print("\n=== Group by: stats per year ===")
for year in sorted(stats):
    s = stats[year]
    avg = round(s["total"] / s["cnt"], 2)
    print(f"  {year}: {s['cnt']:>7,} reviews, avg rating: {avg}")

# 3. window function: top-3 most helpful reviews per year
top_by_year = defaultdict(list)

with open("data.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        year = row["timestamp"][:4]
        vote = int(row["helpful_vote"])
        top = top_by_year[year]
        if len(top) < 3 or vote > top[-1][1]:
            top.append((row["title"], vote))
            top.sort(key=lambda x: -x[1])
            top_by_year[year] = top[:3]

print("\n=== Window: top-3 helpful reviews per year ===")
for year in sorted(top_by_year):
    for rank, (title, vote) in enumerate(top_by_year[year], 1):
        print(f"  {year} | rank {rank} | votes: {vote:>5} | {title[:50]}")

=== Filter: rating=5, year=2022 ===
  A sweet story with a positive message for any child needing   |  2022-02-10 04:01:49.024
  So cute!  |  2022-03-18 04:25:53.809
  Good to the last gory drop!  |  2022-06-19 00:57:19.827
  What rock have I been under?  |  2022-06-13 04:59:16.252
  A short read but jam packed with goodness  |  2022-06-11 05:09:43.375

=== Group by: stats per year ===
  1996:      12 reviews, avg rating: 4.92
  1997:     169 reviews, avg rating: 4.35
  1998:     866 reviews, avg rating: 4.23
  1999:   1,860 reviews, avg rating: 4.21
  2000:   7,315 reviews, avg rating: 4.2
  2001:   7,539 reviews, avg rating: 4.21
  2002:   8,552 reviews, avg rating: 4.24
  2003:   9,696 reviews, avg rating: 4.15
  2004:  11,882 reviews, avg rating: 4.16
  2005:  16,253 reviews, avg rating: 4.11
  2006:  20,009 reviews, avg rating: 4.16
  2007:  26,912 reviews, avg rating: 4.22
  2008:  32,259 reviews, avg rating: 4.17
  2009:  40,443 reviews, avg rating: 4.15
  2010:  53,042 reviews,

### 4.6 Raw Python + оптимизации (Numba JIT + многопоточность)

# prepare numpy arrays for JIT-compiled functions
ratings = df["rating"].values.astype(np.float32)
votes = df["helpful_vote"].values.astype(np.int64)
years = df["timestamp"].dt.year.values.astype(np.int32)


# 1. filter using numba JIT
@njit
def filter_jit(ratings, years, target_rating, target_year):
    indices = []
    for i in range(len(ratings)):
        if ratings[i] == target_rating and years[i] == target_year:
            indices.append(i)
            if len(indices) >= 5:
                break
    return indices


idx = filter_jit(ratings, years, 5.0, 2022)
print("=== Filter (Numba JIT): rating=5, year=2022 ===")
for i in idx:
    print(f"  {df.iloc[i]['title'][:60]}  |  {df.iloc[i]['timestamp']}")


# 2. group by year using numba JIT
@njit
def groupby_jit(ratings, years):
    cnt = np.zeros(2030, dtype=np.int64)
    total = np.zeros(2030, dtype=np.float64)
    for i in range(len(ratings)):
        y = years[i]
        cnt[y] += 1
        total[y] += ratings[i]
    return cnt, total


cnt, total = groupby_jit(ratings, years)
print("\n=== Group by (Numba JIT): stats per year ===")
for y in range(1996, 2024):
    if cnt[y] > 0:
        print(f"  {y}: {cnt[y]:>7,} reviews, avg rating: {round(total[y] / cnt[y], 2)}")


# 3. window function using ThreadPoolExecutor
def top3_for_year(year):
    """Find top-3 most helpful reviews for a given year."""
    mask = years == year
    v = votes[mask]
    idx_local = np.argsort(v)[::-1][:3]
    global_idx = np.where(mask)[0][idx_local]
    return [(df.iloc[i]["title"][:50], int(votes[i]), year) for i in global_idx]


print("\n=== Window (ThreadPool): top-3 helpful reviews per year ===")
year_list = sorted(set(years))
with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    results_top = list(pool.map(top3_for_year, year_list))

for group in results_top:
    for rank, (title, vote, year) in enumerate(group, 1):
        print(f"  {year} | rank {rank} | votes: {vote:>5} | {title}")

## 5. Бенчмарк JSON-парсеров

Сравниваем скорость парсинга JSONL-файла тремя библиотеками: стандартный json, orjson (написан на Rust) и ujson (написан на C).

In [14]:
results_parsers = []


def bench_parser(name, parse_func):
    """Measure parsing speed for a JSON library on the JSONL file."""
    t0_wall = time.time()
    t0_cpu = time.process_time()
    count = 0
    with open("data.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            parse_func(line)
            count += 1
    wall = time.time() - t0_wall
    cpu = time.process_time() - t0_cpu
    results_parsers.append({
        "parser": name,
        "wall_time": round(wall, 2),
        "cpu_time": round(cpu, 2),
    })
    print(f"{name:10s} | wall: {wall:7.2f}s | cpu: {cpu:7.2f}s | rows: {count:,}")


print(f"{'Parser':10s} | {'Wall time':>10s} | {'CPU time':>10s} | {'Rows':>10s}")
print("-" * 55)
bench_parser("json", json.loads)
bench_parser("orjson", orjson.loads)
bench_parser("ujson", ujson.loads)

Parser     |  Wall time |   CPU time |       Rows
-------------------------------------------------------
json       | wall:   16.20s | cpu:   16.23s | rows: 3,000,000
orjson     | wall:    9.18s | cpu:    9.19s | rows: 3,000,000
ujson      | wall:   14.62s | cpu:   14.61s | rows: 3,000,000


**Вывод по парсерам:** orjson (9.2s) быстрее стандартного json (16.2s) почти в 2 раза — он написан на Rust и оптимизирован под SIMD-инструкции. ujson (14.6s) тоже быстрее стандартного, но проигрывает orjson. Для задач, где нужно парсить большие объёмы JSON, orjson — лучший выбор.
## 6. Итоговая сводка

In [17]:
print("=" * 60)
print("SUMMARY")
print(f"Dataset: Amazon Reviews (Books), {TARGET_ROWS:,} rows")
print("=" * 60)

print("\nWRITE BENCHMARK")
print(f"{'Format':10s} | {'Wall':>8s} | {'CPU':>8s} | {'Size':>8s}")
print("-" * 45)
for r in results_write:
    print(f"{r['format']:10s} | {r['wall_time']:>7.2f}s | {r['cpu_time']:>7.2f}s | {r['size_gb']:.2f} GB")

print("\nREAD BENCHMARK")
print(f"{'Format':10s} | {'Wall':>8s} | {'CPU':>8s}")
print("-" * 33)
for r in results_read:
    print(f"{r['format']:10s} | {r['wall_time']:>7.2f}s | {r['cpu_time']:>7.2f}s")

print("\nJSON PARSERS")
print(f"{'Parser':10s} | {'Wall':>8s} | {'CPU':>8s}")
print("-" * 33)
for r in results_parsers:
    print(f"{r['parser']:10s} | {r['wall_time']:>7.2f}s | {r['cpu_time']:>7.2f}s")

print("\nSQL QUERIES EXECUTED WITH:")
libs = [
    "DuckDB", "Polars", "SQLAlchemy (SQLite)", "PyArrow",
    "Raw Python (csv module)", "Raw Python + Numba JIT + ThreadPool",
]
for lib in libs:
    print(f"  - {lib}")

SUMMARY
Dataset: Amazon Reviews (Books), 3,000,000 rows

WRITE BENCHMARK
Format     |     Wall |      CPU |     Size
---------------------------------------------
csv        |   21.12s |   21.11s | 1.54 GB
tsv        |   20.98s |   20.94s | 1.53 GB
json       |   14.56s |   13.53s | 1.82 GB
jsonl      |   25.41s |   24.34s | 1.83 GB
parquet    |    7.52s |    7.50s | 0.87 GB
orc        |    4.19s |    4.06s | 1.45 GB
sqlite     |   38.13s |   18.73s | 1.79 GB

READ BENCHMARK
Format     |     Wall |      CPU
---------------------------------
csv        |   20.04s |   20.05s
tsv        |   20.57s |   20.58s
json       |   22.97s |   23.20s
jsonl      |   31.16s |   31.05s
parquet    |    6.39s |    7.06s
orc        |    7.22s |    5.86s
sqlite     |   11.90s |   10.56s

JSON PARSERS
Parser     |     Wall |      CPU
---------------------------------
json       |   16.20s |   16.23s
orjson     |    9.18s |    9.19s
ujson      |   14.62s |   14.61s

SQL QUERIES EXECUTED WITH:
  - DuckDB
  -

- **Лучший формат для хранения больших данных — Parque.** Самый быстрый на чтение (6.4s) и запись (7.5s), занимает минимум места (0.87 GB vs 1.83 GB у JSON).
- **ORC** -  близок к Parquet по скорости, но занимает больше места (1.45 GB). Менее распространён в Python-экосистеме.
- **CSV/TSV** — универсальные и простые, но медленные (~20-21s на чтение и запись). Подходят для обмена данными, не для аналитики.
- **JSON/JSONL** — самые тяжёлые по размеру (~1.83 GB). Удобны для API и логов, но не для хранения больших таблиц.
- **SQLite** — медленный на запись (38.1s из-за построчных INSERT), но быстрый на чтение (11.9s). Хороший компромисс, если нужны SQL-запросы без сервера.
- **orjson** — лучший JSON-парсер для Python, быстрее стандартного в ~2 раза (9.2s vs 16.2s).
- **DuckDB и Polars** — самые удобные инструменты для аналитических запросов: быстрые, лаконичный синтаксис, работают с файлами напрямую без загрузки в память.

- **Лучший формат для хранения больших данных — Parquet.** Самый быстрый на чтение (6.4s) и запись (7.5s), занимает минимум места (0.87 GB vs 1.83 GB у JSON).
- **ORC** — близок к Parquet по скорости, но занимает больше места (1.45 GB). Менее распространён в Python-экосистеме.
- **CSV/TSV** — универсальные и простые, но медленные (~20-21s на чтение и запись). Подходят для обмена данными, не для аналитики.
- **JSON/JSONL** — самые тяжёлые по размеру (~1.83 GB). Удобны для API и логов, но не для хранения больших таблиц.
- **SQLite** — медленный на запись (38.1s из-за построчных INSERT), но быстрый на чтение (11.9s). Хороший компромисс, если нужны SQL-запросы без сервера.
- **orjson** — лучший JSON-парсер для Python, быстрее стандартного в ~2 раза (9.2s vs 16.2s).
- **DuckDB и Polars** — самые удобные инструменты для аналитических запросов: быстрые, лаконичный синтаксис, работают с файлами напрямую без загрузки в память.